# CDR-MLC Causal Soft Routing

This experiment preserves the leakage-safe CDR-MLC training pipeline and compares:

1. **Hard routing**: the nearest K-Means congestion expert only.
2. **Top-2 distance routing**: inverse-distance weighted class probabilities from the two nearest experts.
3. **Top-2 distance-confidence routing**: the same weights multiplied by each expert's maximum class probability.

The router uses only the same 15 causal features (3 signals × 5 statistics, window size 3). Test labels are accessed only after all predictions have been produced, exclusively for evaluation. No test label, Oracle target, test-time fitting, or global classifier is used.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

MAIN_NOTEBOOK = Path("CDR-MLC.ipynb")
if not MAIN_NOTEBOOK.exists():
    MAIN_NOTEBOOK = Path("CDR_MLC") / "CDR-MLC.ipynb"

with MAIN_NOTEBOOK.open(encoding="utf-8") as handle:
    main_notebook = json.load(handle)

namespace = {}
exec(compile(
    "".join(main_notebook["cells"][0]["source"]),
    str(MAIN_NOTEBOOK),
    "exec",
), namespace)

run_pipeline_from_two_files = namespace["run_pipeline_from_two_files"]
compute_sliding_window_stats = namespace["compute_sliding_window_stats"]

print("Loaded the exact leakage-safe CDR-MLC implementation")


In [ ]:
def _aligned_probabilities(classifier, X, global_classes):
    """Place an expert's predict_proba columns in the shared class space."""
    raw = classifier.predict_proba(X)
    aligned = np.zeros((len(X), len(global_classes)), dtype=np.float64)
    class_position = {label: index for index, label in enumerate(global_classes)}
    for source_column, label in enumerate(classifier.classes_):
        aligned[:, class_position[label]] = raw[:, source_column]
    return aligned


def _metric_row(method, y_true, y_pred):
    return {
        "method": method,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_weighted": precision_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted"),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
    }


def evaluate_causal_soft_routing(scenario, train_file, test_file):
    """
    Fit the unchanged leakage-safe CDR-MLC pipeline and evaluate causal soft gates.

    Important ordering:
      1. train models;
      2. construct all routes, weights, probabilities, and predictions without y_test;
      3. access y_test only to calculate final metrics.
    """
    print(f"\n{'=' * 80}\n{scenario}\n{'=' * 80}")
    result = run_pipeline_from_two_files(
        train_file=train_file,
        test_file=test_file,
        n_clusters=3,
        window_size=3,
        clustering_stats=["mean", "median", "std", "min", "max"],
    )

    test_df = result["test_df"]
    feature_columns = result["classification_features"]
    experts = result["classifiers"]
    expert_ids = np.array(sorted(experts))
    X_test = test_df[feature_columns].to_numpy()

    # Reconstruct the exact causal 15-D routing vector. This uses input features only.
    routing_stats, _ = compute_sliding_window_stats(
        test_df[["SynAck", "AckDat", "TcpRtt"]],
        ["SynAck", "AckDat", "TcpRtt"],
        result["window_size"],
        result["clustering_stats"],
    )
    scaled_routing = result["scaler"].transform(routing_stats)
    distances = result["kmeans_model"].transform(scaled_routing)[:, expert_ids]

    global_classes = np.arange(len(result["label_encoder"].classes_))
    expert_probabilities = np.stack(
        [
            _aligned_probabilities(experts[expert_id], X_test, global_classes)
            for expert_id in expert_ids
        ],
        axis=1,
    )

    # Existing hard route, vectorized.
    hard_routes = test_df["cluster"].to_numpy()
    hard_predictions = np.empty(len(X_test), dtype=int)
    for expert_id in expert_ids:
        mask = hard_routes == expert_id
        hard_predictions[mask] = experts[expert_id].predict(X_test[mask])

    # Two nearest congestion experts.
    top2_columns = np.argsort(distances, axis=1)[:, :2]
    rows = np.arange(len(X_test))[:, None]
    top2_distances = np.take_along_axis(distances, top2_columns, axis=1)
    top2_probabilities = expert_probabilities[rows, top2_columns, :]

    inverse_distance = 1.0 / np.maximum(top2_distances, 1e-12)
    distance_weights = inverse_distance / inverse_distance.sum(axis=1, keepdims=True)
    distance_predictions = (
        top2_probabilities * distance_weights[:, :, None]
    ).sum(axis=1).argmax(axis=1)

    expert_confidence = top2_probabilities.max(axis=2)
    confidence_weights = inverse_distance * expert_confidence
    confidence_weights /= np.maximum(
        confidence_weights.sum(axis=1, keepdims=True), 1e-12
    )
    confidence_predictions = (
        top2_probabilities * confidence_weights[:, :, None]
    ).sum(axis=1).argmax(axis=1)

    # Evaluation boundary: test labels first become visible here.
    y_test = test_df[result["target_column"]].to_numpy()
    metrics = pd.DataFrame([
        _metric_row("hard_kmeans", y_test, hard_predictions),
        _metric_row("top2_distance", y_test, distance_predictions),
        _metric_row(
            "top2_distance_confidence",
            y_test,
            confidence_predictions,
        ),
    ])
    baseline = metrics.loc[metrics["method"] == "hard_kmeans", "accuracy"].iloc[0]
    metrics["accuracy_gain_pp"] = 100 * (metrics["accuracy"] - baseline)

    route_counts = pd.Series(hard_routes).value_counts().sort_index()
    route_distribution = pd.DataFrame({
        "expert": route_counts.index.astype(int),
        "routed_samples": route_counts.values,
        "route_share_%": 100 * route_counts.values / len(hard_routes),
    })

    display(metrics.round(4))
    display(route_distribution.round(4))
    return {
        "training_result": result,
        "metrics": metrics,
        "route_distribution": route_distribution,
        "predictions": {
            "hard_kmeans": hard_predictions,
            "top2_distance": distance_predictions,
            "top2_distance_confidence": confidence_predictions,
        },
    }


In [ ]:
# scenario_1: run independently
scenario_1_soft = evaluate_causal_soft_routing(
    "scenario_1",
    "DATASETS/CDR-MLC/scale_1/Short/level_1.csv",
    "DATASETS/CDR-MLC/scale_1/Short/level_2.csv",
)


In [ ]:
# scenario_2: run independently
scenario_2_soft = evaluate_causal_soft_routing(
    "scenario_2",
    "DATASETS/CDR-MLC/scale_1/Short/level_1.csv",
    "DATASETS/CDR-MLC/scale_1/Short/level_3.csv",
)


In [ ]:
# scenario_3: run independently
scenario_3_soft = evaluate_causal_soft_routing(
    "scenario_3",
    "DATASETS/CDR-MLC/scale_1/Short/level_2.csv",
    "DATASETS/CDR-MLC/scale_1/Short/level_3.csv",
)


In [ ]:
# scenario_4: run independently
scenario_4_soft = evaluate_causal_soft_routing(
    "scenario_4",
    "DATASETS/CDR-MLC/scale_1/Short/CDR-MLC-Shuffle.csv",
    "DATASETS/CDR-MLC/scale_1/Long/CDR-MLC-Shuffle.csv",
)


In [ ]:
# scenario_5: run independently
scenario_5_soft = evaluate_causal_soft_routing(
    "scenario_5",
    "DATASETS/CDR-MLC/scale_1/Long/CDR-MLC-Shuffle.csv",
    "DATASETS/CDR-MLC/scale_1/Short/CDR-MLC-Shuffle.csv",
)
